In [0]:
SELECT * FROM workspace.silver.voltas limit 10

In [0]:
%sql
create or replace table workspace.gold.desempenho_piloto AS    
WITH melhores_voltas AS (
    SELECT
        chave_reuniao,
        chave_sessao,
        numero_piloto,
        MIN(duracao_volta) AS melhor_duracao_volta
    FROM workspace.silver.voltas
    GROUP BY chave_reuniao, chave_sessao, numero_piloto
),
voltas_com_melhor AS (
    SELECT 
        v.*,
        m.melhor_duracao_volta
    FROM workspace.silver.voltas v
    JOIN melhores_voltas m
        ON v.chave_reuniao = m.chave_reuniao
        AND v.chave_sessao = m.chave_sessao
        AND v.numero_piloto = m.numero_piloto
    WHERE v.duracao_volta IS NOT NULL
      AND v.volta_saida_pit = FALSE
)

SELECT
    chave_reuniao,
    chave_sessao,
    numero_piloto,

    COUNT(numero_volta) AS total_voltas,
    MIN(duracao_volta) AS melhor_duracao_volta,
    ROUND(AVG(duracao_volta), 3) AS media_duracao_volta,
    ROUND(STDDEV(duracao_volta), 3) AS desvio_padrao_volta,
    ROUND(AVG(duracao_volta) - MIN(duracao_volta), 3) AS media_gap_para_melhor,

    MIN(duracao_setor_1) AS melhor_setor_1,
    MIN(duracao_setor_2) AS melhor_setor_2,
    MIN(duracao_setor_3) AS melhor_setor_3,

    ROUND(AVG(duracao_setor_1), 3) AS media_setor_1,
    ROUND(AVG(duracao_setor_2), 3) AS media_setor_2,
    ROUND(AVG(duracao_setor_3), 3) AS media_setor_3,

    MAX(velocidade_armadilha) AS max_velocidade_reta,
    ROUND(AVG(velocidade_armadilha), 1) AS media_velocidade_reta,

    SUM(CASE WHEN volta_saida_pit THEN 1 ELSE 0 END) AS paradas_pit,

    ROUND(
        SUM(CASE WHEN duracao_volta <= melhor_duracao_volta * 1.05 THEN 1 ELSE 0 END) * 100.0 
        / COUNT(numero_volta), 1
    ) AS pct_voltas_dentro_105,

    MIN(data_inicio) AS inicio_sessao,
    MAX(data_inicio) AS fim_sessao,
    current_timestamp() AS _processado_em

FROM voltas_com_melhor
GROUP BY chave_reuniao, chave_sessao, numero_piloto;



In [0]:
select * from workspace.gold.desempenho_piloto